# Notebook 12c — MQAR Capacity at the Real Roadmap Config
## Hetero-VLA (4x32 fast + 4x256 slow = 1152 cap) vs Uniform-VLA (8x128 = 1024 cap)

### What changed from notebook 12b
The previous "Fast" vectorized models had a real architecture bug: `Wu`
(the penalty-direction projection) was applied to the FULL `d_model` input
via a single global `Linear(d_model, d_model)`, then split into heads --
instead of being applied to each head's OWN raw input slice via a per-head
`Linear(dh, dh)`, which is what the validated single-head class and every
successful earlier experiment actually used. This silently changed the
model (mixing information across head boundaries before head-specific
processing) and is very likely why training stalled at exactly random
chance for harder configs (larger d_h, larger T).

This notebook uses `VLAv3MultiHeadCorrect`: per-head raw-slice input,
per-head batched-einsum projections (Wq/Wk/Wv/Wu), verified bit-exact
against the original validated reference architecture (max diff = 0.000e+00,
see the verification cell below). Only `Wo` (the final output mixing after
heads are concatenated) is a global projection -- that was always correct.

### The experiment
| Model | Heads | Capacity | d_model |
|---|---|---|---|
| Uniform-VLA | 8 x d_h=128 | 1024 | 1024 |
| Hetero-VLA | 4 x d_h=32 (fast) + 4 x d_h=256 (slow) | 1152 | 1152 |
| DeltaNet | 8 x d_h=128 | 1024 | 1024 |

n_pairs swept: 32, 64, 96, 128, 160, 200, 256 (200+ as required).
3 seeds. Chunked gradient checkpointing (fp32, no AMP, verified bit-exact
vs unchunked) keeps memory bounded at d_h=256 regardless of T.

### Pass criteria
Hetero-VLA accuracy > 0.80 at n_pairs=200. Uniform-VLA collapses well
before that (expected near its d_h=128 boundary). This is the one
experiment that actually answers the roadmap's central capacity claim.

## 0 · Setup

In [ ]:
import math, time, gc, json, glob
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
OUT    = Path('/kaggle/working/nb12c_real_capacity')
for sub in ['plots', 'logs']:
    (OUT/sub).mkdir(parents=True, exist_ok=True)

matplotlib.rcParams.update({
    'font.family' : 'DejaVu Serif', 'font.size' : 11,
    'axes.spines.top' : False, 'axes.spines.right' : False,
    'axes.grid' : True, 'grid.alpha' : 0.25, 'figure.dpi' : 150,
})
C = {'uniform_vla':'#55EFC4', 'hetero_vla':'#00B894', 'deltanet':'#FDCB6E'}
M = {'uniform_vla':'o', 'hetero_vla':'D', 'deltanet':'^'}

print(f'Device : {DEVICE}')
print(f'Torch  : {torch.__version__}')
print(f'Output : {OUT}')

# ── cap_rows recovery: survives kernel restarts between sweep cells ──────
def _recover_cap_rows():
    files = sorted(glob.glob(str(OUT/'logs'/'exp1_n*.csv')))
    rows = []
    for fp in files:
        rows.extend(pd.read_csv(fp).to_dict('records'))
    return rows

if 'cap_rows' not in dir() or not isinstance(cap_rows, list):
    cap_rows = _recover_cap_rows()
    if cap_rows:
        done_n = sorted(set(r['n_pairs'] for r in cap_rows))
        print(f'Recovered {len(cap_rows)} rows from disk for n_pairs={done_n}')
    else:
        print('No checkpoint files found -- starting cap_rows fresh.')

## 1 · Real Roadmap Config

In [ ]:
# ── Heterogeneous head design -- THE REAL ROADMAP CONFIG ─────────────────
FAST_H, FAST_DH = 4, 32     # fast heads: recent associations
SLOW_H, SLOW_DH = 4, 256    # slow heads: long-range, large capacity
D_HETERO   = FAST_H*FAST_DH + SLOW_H*SLOW_DH   # 128 + 1024 = 1152
CAP_HETERO = FAST_H*FAST_DH + SLOW_H*SLOW_DH   # 1152

# ── Uniform comparison ─────────────────────────────────────────────────────
UNIF_H, UNIF_DH = 8, 128
D_UNIF   = UNIF_H * UNIF_DH    # 1024
CAP_UNIF = D_UNIF              # 1024

# ── MQAR task ──────────────────────────────────────────────────────────────
VOCAB = 512   # > max(n_pairs)+1, gives 511 unique keys available
N_PAIRS = [32, 64, 96, 128, 160, 200, 256]   # 200+ as required by roadmap

# ── Training ───────────────────────────────────────────────────────────────
SEEDS  = [42, 123, 999]
BATCH  = 32     # safe: checkpointing bounds memory at O(chunk_size), not O(T)
LR     = 1e-3
WARMUP = 150
CHECKPOINT_CHUNK = 32   # verified bit-exact vs unchunked at multiple sizes

def steps_for_n(n):
    # Associative-recall tasks can sit flat near random for hundreds of
    # steps before a sudden jump to high accuracy. Harder/larger configs
    # push that transition later -- budget generously.
    if n <= 64:  return 2000
    if n <= 128: return 2800
    return 3200

print('REAL ROADMAP CONFIG:')
print(f'  Hetero-VLA : {FAST_H}x d_h={FAST_DH} (fast) + {SLOW_H}x d_h={SLOW_DH} (slow)')
print(f'               d_model={D_HETERO}  capacity={CAP_HETERO}')
print(f'  Uniform-VLA: {UNIF_H}x d_h={UNIF_DH}')
print(f'               d_model={D_UNIF}  capacity={CAP_UNIF}')
print()
print(f'n_pairs sweep: {N_PAIRS}')
print(f'Seeds: {SEEDS}')
for n in N_PAIRS:
    print(f'  n={n:4d}: {steps_for_n(n):,} steps')

## 2 · Corrected Model Definitions

`VLAv3MultiHeadCorrect`: per-head raw input slice, per-head batched-einsum
projections. Verified bit-exact against the validated reference (see the
verification cell immediately below) -- this is the bug fix from notebook 12b.

Chunked gradient checkpointing applied to the T-loop only (the upfront
Wq/Wk/Wv/Wu/U projections are computed once for the full sequence, exactly
as in the reference; only the sequential SM/S recurrence is chunked).
fp32 throughout, no AMP -- verified bit-exact vs unchunked at multiple
chunk sizes including ones that don't evenly divide T or align with the
SM periodic-refresh schedule.

In [ ]:
import torch.utils.checkpoint as torch_checkpoint

def _split_raw(x, H):
    """Raw per-head slice, NO projection -- matches x[:,:,i*dh:(i+1)*dh]."""
    B, T, D = x.shape
    dh = D // H
    return x.view(B, T, H, dh).permute(0, 2, 1, 3)   # (B,H,T,dh)

def _merge(x):
    """(B,H,T,dh) -> (B,T,D)"""
    B, H, T, dh = x.shape
    return x.permute(0, 2, 1, 3).reshape(B, T, H*dh)


def _vla_chunk_step(kf_c, Q_c, V_c, U_c, A, S, zk, I, isq, t_offset, eps, period, per_eps):
    # One checkpointed chunk of the recurrence. Pure function of its inputs.
    # t_offset is the GLOBAL starting timestep, so the periodic identity
    # refresh lands on the same global steps regardless of chunk boundaries.
    B, H = A.shape[0], A.shape[1]
    Tc = kf_c.shape[2]
    ys = []
    for tc in range(Tc):
        t = t_offset + tc
        u   = U_c[:,:,tc,:] * isq
        zsm = torch.einsum('bhde,bhe->bhd', A, u)
        dlt = (1.0 + (u*zsm).sum(-1)).clamp(min=eps)
        A   = A - torch.einsum('bhd,bhe->bhde', zsm, zsm) / dlt.view(B,H,1,1)
        if (t+1) % period == 0:
            A = A + per_eps * I
        kn    = F.normalize(kf_c[:,:,tc,:], p=2, dim=-1)
        alpha = torch.einsum('bhde,bhe->bhd', A, kn)
        alphn = F.normalize(alpha, p=2, dim=-1)
        e     = V_c[:,:,tc,:] - torch.einsum('bhde,bhe->bhd', S, kn)
        S     = S + torch.einsum('bhd,bhe->bhde', e, alphn)
        qt    = Q_c[:,:,tc,:]
        zk    = zk + kf_c[:,:,tc,:]
        yt    = torch.einsum('bhde,bhe->bhd', S, qt)
        ys.append(yt / (zk*qt).sum(-1,keepdim=True).clamp(min=eps))
    return torch.stack(ys, dim=2), A, S, zk


class VLAv3MultiHeadCorrect(nn.Module):
    """
    Vectorized multi-head VLAv3, architecturally faithful to the single-head
    validated reference: each head's Wq/Wk/Wv/Wu act on that head's OWN raw
    input slice (no global d_model x d_model mixing before head-splitting).
    Wu is applied to kr (the head's key projection), not to raw x -- this
    is the corrected line; the bug applied Wu to x directly.
    """
    def __init__(self, d_model, H, lam=0.1, eps=1e-4, per_eps=1e-3, period=20,
                 chunk=None):
        super().__init__()
        assert d_model % H == 0
        self.H = H; self.dh = d_model // H
        self.lam=lam; self.eps=eps; self.per_eps=per_eps; self.period=period
        self.chunk = chunk if chunk is not None else CHECKPOINT_CHUNK
        dh = self.dh
        self.Wq_w = nn.Parameter(torch.empty(H, dh, dh)); self.Wq_b = nn.Parameter(torch.zeros(H, dh))
        self.Wk_w = nn.Parameter(torch.empty(H, dh, dh)); self.Wk_b = nn.Parameter(torch.zeros(H, dh))
        self.Wv_w = nn.Parameter(torch.empty(H, dh, dh)); self.Wv_b = nn.Parameter(torch.zeros(H, dh))
        self.Wu_w = nn.Parameter(torch.empty(H, dh, dh))   # bias=False, matches spec
        for w in [self.Wq_w, self.Wk_w, self.Wv_w, self.Wu_w]:
            nn.init.kaiming_uniform_(w, a=math.sqrt(5))
        self.Wo   = nn.Linear(d_model, d_model)   # global output mixing -- correct, unaffected
        self.norm = nn.LayerNorm(d_model)

    def capacity(self): return self.H * self.dh

    def forward(self, x):
        B, T, D = x.shape; H, dh = self.H, self.dh
        xh  = _split_raw(x, H)                                              # (B,H,T,dh)
        kr  = torch.einsum('bhtd,hde->bhte', xh, self.Wk_w) + self.Wk_b.view(1,H,1,dh)
        kf  = F.elu(kr) + 1.0
        Q   = F.elu(torch.einsum('bhtd,hde->bhte', xh, self.Wq_w) + self.Wq_b.view(1,H,1,dh)) + 1.0
        V   = torch.einsum('bhtd,hde->bhte', xh, self.Wv_w) + self.Wv_b.view(1,H,1,dh)
        U   = F.normalize(torch.einsum('bhtd,hde->bhte', kr, self.Wu_w), p=2, dim=-1)   # Wu(kr) -- CORRECT

        I   = torch.eye(dh, device=x.device, dtype=x.dtype)
        A   = (1/self.lam) * I.view(1,1,dh,dh).expand(B,H,-1,-1).clone()
        S   = torch.zeros(B, H, dh, dh, device=x.device, dtype=x.dtype)
        zk  = torch.zeros(B, H, dh, device=x.device, dtype=x.dtype)
        isq = 1.0 / math.sqrt(dh)

        ys = []
        for start in range(0, T, self.chunk):
            end = min(start + self.chunk, T)
            out_c, A, S, zk = torch_checkpoint.checkpoint(
                _vla_chunk_step,
                kf[:,:,start:end], Q[:,:,start:end], V[:,:,start:end], U[:,:,start:end],
                A, S, zk, I, isq, start, self.eps, self.period, self.per_eps,
                use_reentrant=False)
            ys.append(out_c)
        out = torch.cat(ys, dim=2)
        return self.Wo(self.norm(_merge(out)))


def _deltanet_chunk_step(kt_c, vt_c, qt_c, gt_c, S):
    Tc = kt_c.shape[2]
    ys = []
    for tc in range(Tc):
        kt, vt, qt, gt = kt_c[:,:,tc,:], vt_c[:,:,tc,:], qt_c[:,:,tc,:], gt_c[:,:,tc,:]
        pred = torch.einsum('bhde,bhe->bhd', S, kt)
        S = gt.unsqueeze(-1) * S + torch.einsum('bhd,bhe->bhde', vt - pred, kt)
        ys.append(torch.einsum('bhde,bhe->bhd', S, qt))
    return torch.stack(ys, dim=2), S


class DeltaNetMultiHeadCorrect(nn.Module):
    """DeltaNet baseline, same per-head raw-slice convention as VLA."""
    def __init__(self, d_model, H, eps=1e-6, chunk=None):
        super().__init__()
        self.H=H; self.dh=d_model//H; self.eps=eps
        self.chunk = chunk if chunk is not None else CHECKPOINT_CHUNK
        dh = self.dh
        self.Wq_w = nn.Parameter(torch.empty(H, dh, dh)); self.Wq_b = nn.Parameter(torch.zeros(H, dh))
        self.Wk_w = nn.Parameter(torch.empty(H, dh, dh)); self.Wk_b = nn.Parameter(torch.zeros(H, dh))
        self.Wv_w = nn.Parameter(torch.empty(H, dh, dh)); self.Wv_b = nn.Parameter(torch.zeros(H, dh))
        self.Wg_w = nn.Parameter(torch.empty(H, dh, dh)); self.Wg_b = nn.Parameter(torch.zeros(H, dh))
        for w in [self.Wq_w, self.Wk_w, self.Wv_w, self.Wg_w]:
            nn.init.kaiming_uniform_(w, a=math.sqrt(5))
        self.Wo = nn.Linear(d_model, d_model)

    def forward(self, x):
        B, T, D = x.shape; H, dh = self.H, self.dh
        xh = _split_raw(x, H)
        Q  = F.elu(torch.einsum('bhtd,hde->bhte', xh, self.Wq_w) + self.Wq_b.view(1,H,1,dh)) + 1.0
        K  = F.normalize(torch.einsum('bhtd,hde->bhte', xh, self.Wk_w) + self.Wk_b.view(1,H,1,dh), p=2, dim=-1)
        V  = torch.einsum('bhtd,hde->bhte', xh, self.Wv_w) + self.Wv_b.view(1,H,1,dh)
        G  = torch.sigmoid(torch.einsum('bhtd,hde->bhte', xh, self.Wg_w) + self.Wg_b.view(1,H,1,dh))

        S = torch.zeros(B, H, dh, dh, device=x.device, dtype=x.dtype)
        ys = []
        for start in range(0, T, self.chunk):
            end = min(start + self.chunk, T)
            out_c, S = torch_checkpoint.checkpoint(
                _deltanet_chunk_step,
                K[:,:,start:end], V[:,:,start:end], Q[:,:,start:end], G[:,:,start:end], S,
                use_reentrant=False)
            ys.append(out_c)
        out = torch.cat(ys, dim=2)
        return self.Wo(_merge(out))


class HeteroVLACorrect(nn.Module):
    """Fast group + slow group, each the corrected per-head architecture."""
    def __init__(self, fast_H=FAST_H, fast_dh=FAST_DH,
                       slow_H=SLOW_H, slow_dh=SLOW_DH):
        super().__init__()
        self.fast = VLAv3MultiHeadCorrect(fast_H*fast_dh, fast_H)
        self.slow = VLAv3MultiHeadCorrect(slow_H*slow_dh, slow_H)
        self._fast_d = fast_H*fast_dh
        self._d = fast_H*fast_dh + slow_H*slow_dh
        self.Wo = nn.Linear(self._d, self._d)
        self.norm = nn.LayerNorm(self._d)

    def capacity(self): return self.fast.capacity() + self.slow.capacity()

    def forward(self, x):
        xf = x[:, :, :self._fast_d]
        xs = x[:, :, self._fast_d:]
        return self.Wo(self.norm(
            torch.cat([self.fast(xf), self.slow(xs)], dim=-1)))


class Block(nn.Module):
    def __init__(self, attn, d, ff_mult=2):
        super().__init__()
        self.ln1 = nn.LayerNorm(d); self.ln2 = nn.LayerNorm(d)
        self.attn = attn
        self.ff = nn.Sequential(
            nn.Linear(d, d*ff_mult), nn.GELU(), nn.Linear(d*ff_mult, d))
    def forward(self, x):
        return x + self.ff(self.ln2(x + self.attn(self.ln1(x))))


class TinyLM(nn.Module):
    def __init__(self, attn_fn, d, vocab=VOCAB, n_layers=2):
        super().__init__()
        self.tok = nn.Embedding(vocab, d)
        self.pos = nn.Embedding(8192, d)
        self.blocks = nn.ModuleList([Block(attn_fn(d), d) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d)
        self.head = nn.Linear(d, vocab, bias=False)
        self.head.weight = self.tok.weight
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.02)
                if m.bias is not None: nn.init.zeros_(m.bias)
            elif isinstance(m, nn.Embedding):
                nn.init.normal_(m.weight, std=0.02)

    def forward(self, idx):
        B, T = idx.shape
        x = (self.tok(idx) +
             self.pos(torch.arange(T, device=idx.device).unsqueeze(0)))
        for b in self.blocks: x = b(x)
        return self.ln_f(x) @ self.tok.weight.T


# ── Registry -- the real roadmap config ───────────────────────────────────
MODEL_REGISTRY = {
    'Uniform-VLA': (lambda d: VLAv3MultiHeadCorrect(d, H=UNIF_H),
                    D_UNIF, C['uniform_vla'], M['uniform_vla']),
    'Hetero-VLA':  (lambda d: HeteroVLACorrect(),
                    D_HETERO, C['hetero_vla'], M['hetero_vla']),
    'DeltaNet':    (lambda d: DeltaNetMultiHeadCorrect(d, H=UNIF_H),
                    D_UNIF, C['deltanet'], M['deltanet']),
}

print('NaN checks:')
for name, (factory, d_model, _, _) in MODEL_REGISTRY.items():
    m = TinyLM(factory, d_model, VOCAB).to(DEVICE)
    x = torch.randint(0, VOCAB, (2, 32), device=DEVICE)
    o = m(x)
    p = sum(q.numel() for q in m.parameters())/1e6
    print(f'  {name:14s}: NaN={torch.isnan(o).any().item()}  params={p:.2f}M  d_model={d_model}')
    del m; gc.collect()

## 3 · Verification — Corrected Model Matches the Validated Reference

This is the bug-fix proof, runnable here so you can confirm it yourself
rather than taking it on faith. Builds the original validated single-head
reference, copies its weights into the corrected fast model, and checks
the outputs are bit-identical. Then confirms chunked checkpointing doesn't
change anything either.

In [ ]:
print('VERIFICATION 1: VLAv3MultiHeadCorrect matches the validated reference')
print('='*65)

class VLAv3HeadRef(nn.Module):
    """The original validated single-head class (from the BPC notebooks)."""
    def __init__(self, dh, lam=0.1, eps=1e-4, per_eps=1e-3, period=20):
        super().__init__()
        self.dh=dh; self.lam=lam; self.eps=eps; self.per_eps=per_eps; self.period=period
        self.Wq=nn.Linear(dh,dh); self.Wk=nn.Linear(dh,dh)
        self.Wv=nn.Linear(dh,dh); self.Wu=nn.Linear(dh,dh,bias=False)
    def forward(self, x):
        B,T,d = x.shape
        kr=self.Wk(x); kf=F.elu(kr)+1.0
        Q=F.elu(self.Wq(x))+1.0; V=self.Wv(x)
        U=F.normalize(self.Wu(kr),p=2,dim=-1)
        I=torch.eye(d,device=x.device,dtype=x.dtype)
        A=(1/self.lam)*I.unsqueeze(0).expand(B,-1,-1).clone()
        S=torch.zeros(B,d,d,device=x.device,dtype=x.dtype)
        zk=torch.zeros(B,d,device=x.device,dtype=x.dtype)
        isq=1/math.sqrt(d); ys=[]
        for t in range(T):
            u=U[:,t,:]*isq
            zsm=torch.bmm(A,u.unsqueeze(-1)).squeeze(-1)
            dlt=(1+(u*zsm).sum(-1)).clamp(min=self.eps)
            A=A-torch.einsum('bi,bj->bij',zsm,zsm)/dlt.view(B,1,1)
            if (t+1)%self.period==0: A=A+self.per_eps*I.unsqueeze(0)
            kn=F.normalize(kf[:,t,:],p=2,dim=-1)
            alpha=torch.bmm(A,kn.unsqueeze(-1)).squeeze(-1)
            alphn=F.normalize(alpha,p=2,dim=-1)
            e=V[:,t,:]-torch.bmm(S,kn.unsqueeze(-1)).squeeze(-1)
            S=S+torch.einsum('bi,bj->bij',e,alphn)
            qt=Q[:,t,:]; zk=zk+kf[:,t,:]
            yt=torch.bmm(S,qt.unsqueeze(-1)).squeeze(-1)
            ys.append(yt/(zk*qt).sum(-1,keepdim=True).clamp(min=self.eps))
        return torch.stack(ys,1)

class RefMultiHead(nn.Module):
    def __init__(self, d_model, H):
        super().__init__()
        self.H=H; dh=d_model//H
        self.heads=nn.ModuleList([VLAv3HeadRef(dh) for _ in range(H)])
        self.Wo=nn.Linear(d_model,d_model); self.norm=nn.LayerNorm(d_model)
    def forward(self, x):
        B,T,D=x.shape; dh=D//self.H; outs=[]
        for i,h in enumerate(self.heads):
            outs.append(h(x[:,:,i*dh:(i+1)*dh]))
        return self.Wo(self.norm(torch.cat(outs,-1)))

torch.manual_seed(0)
d_model_v, H_v = 64, 4
ref  = RefMultiHead(d_model_v, H_v).double()
fast = VLAv3MultiHeadCorrect(d_model_v, H_v, chunk=7).double()  # odd chunk size on purpose

with torch.no_grad():
    for h in range(H_v):
        fast.Wq_w[h] = ref.heads[h].Wq.weight.T; fast.Wq_b[h] = ref.heads[h].Wq.bias
        fast.Wk_w[h] = ref.heads[h].Wk.weight.T; fast.Wk_b[h] = ref.heads[h].Wk.bias
        fast.Wv_w[h] = ref.heads[h].Wv.weight.T; fast.Wv_b[h] = ref.heads[h].Wv.bias
        fast.Wu_w[h] = ref.heads[h].Wu.weight.T
    fast.Wo.weight.copy_(ref.Wo.weight); fast.Wo.bias.copy_(ref.Wo.bias)
    fast.norm.weight.copy_(ref.norm.weight); fast.norm.bias.copy_(ref.norm.bias)

x_v = torch.randn(2, 23, d_model_v, dtype=torch.float64)   # T=23, not a multiple of chunk=7
out_ref  = ref(x_v)
out_fast = fast(x_v)
diff = (out_ref - out_fast).abs().max().item()
print(f'  Reference vs corrected (weights matched, T=23, chunk=7): max diff = {diff:.3e}')
print(f'  {"PASS" if diff < 1e-9 else "FAIL"}')

print()
print('VERIFICATION 2: chunk size does not change the result (memory-only tradeoff)')
print('='*65)
for chunk_size in [1, 5, 23, 50]:
    torch.manual_seed(1)
    m1 = VLAv3MultiHeadCorrect(d_model_v, H_v, chunk=chunk_size).double()
    torch.manual_seed(1)
    m2 = VLAv3MultiHeadCorrect(d_model_v, H_v, chunk=100).double()  # effectively unchunked
    out1 = m1(x_v); out2 = m2(x_v)
    d = (out1-out2).abs().max().item()
    print(f'  chunk={chunk_size:4d} vs chunk=100: max diff = {d:.3e}  {"PASS" if d<1e-9 else "FAIL"}')

print()
print('Both verifications must show PASS before trusting the sweep below.')

## 4 · MQAR Task Builder

In [ ]:
def make_mqar(B, n_pairs, vocab=VOCAB, device=DEVICE):
    sep       = vocab - 1
    key_range = max(vocab - 1, n_pairs + 1)
    T         = 2*n_pairs + 1 + n_pairs
    x = torch.full((B, T), sep, dtype=torch.long, device=device)
    y = torch.full((B, T), -100, dtype=torch.long, device=device)
    for b in range(B):
        raw  = torch.randperm(key_range, device=device)[:n_pairs]
        keys = raw % (vocab - 1)
        vals = torch.randint(0, vocab - 1, (n_pairs,), device=device)
        for i in range(n_pairs):
            x[b, 2*i]   = keys[i]
            x[b, 2*i+1] = vals[i]
        x[b, 2*n_pairs] = sep
        perm = torch.randperm(n_pairs, device=device)
        x[b, 2*n_pairs+1:] = keys[perm]
        y[b, 2*n_pairs+1:] = vals[perm]
    return x, y


def run_mqar(attn_fn, d_model, n_pairs,
             steps=2000, batch=BATCH, lr=LR, warmup=WARMUP, seed=42,
             log_every=300, verbose=True):
    # Plain fp32 training. No AMP. Sherman-Morrison subtracts two
    # nearly-equal quantities every step -- numerically sensitive;
    # float32 throughout is load-bearing, not optional.
    torch.manual_seed(seed)
    model = TinyLM(attn_fn, d_model, VOCAB).to(DEVICE)
    opt   = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    def lrf(s):
        if s < warmup: return s / max(warmup, 1)
        return 0.5 * (1 + math.cos(math.pi*(s-warmup)/max(steps-warmup,1)))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lrf)

    rand_baseline = 1/(VOCAB-1)
    t0 = time.time()
    model.train()
    for step in range(1, steps + 1):
        x, y = make_mqar(batch, n_pairs, VOCAB)
        loss = F.cross_entropy(
            model(x).view(-1, VOCAB), y.view(-1), ignore_index=-100)
        if not torch.isfinite(loss):
            if verbose: print(f'      [non-finite loss at step {step} -- stopping]')
            break
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step(); sched.step()

        if verbose and (step % log_every == 0 or step == 1):
            model.eval()
            with torch.no_grad():
                pa = []
                for _ in range(2):
                    xp, yp = make_mqar(32, n_pairs, VOCAB)
                    mp = yp != -100
                    if mp.any():
                        pa.append((model(xp).argmax(-1)[mp]==yp[mp]).float().mean().item())
                probe_acc = float(np.mean(pa)) if pa else 0.0
            model.train()
            flag = '  <- still ~random' if probe_acc < rand_baseline*3 else ''
            print(f'      step {step:5d}/{steps}  loss={loss.item():.4f}  '
                  f'probe_acc={probe_acc:.4f}  ({time.time()-t0:.0f}s){flag}')

    model.eval(); accs = []
    with torch.no_grad():
        for _ in range(20):
            xv, yv = make_mqar(64, n_pairs, VOCAB)
            mask   = yv != -100
            if mask.any():
                accs.append(
                    (model(xv).argmax(-1)[mask] == yv[mask]).float().mean().item())
    del model; gc.collect()
    if DEVICE == 'cuda': torch.cuda.empty_cache()
    return float(np.mean(accs)) if accs else 0.0


print('MQAR sanity checks:')
for n, v in [(8,512),(64,512),(128,512),(200,512),(256,512)]:
    x, y = make_mqar(4, n, v)
    assert x.max().item() <= v-1
    assert (y != -100).sum().item() == 4*n
    print(f'  n={n:4d}  T={x.shape[1]:4d}  targets={4*n}  OK')
print(f'  Random baseline = 1/{VOCAB-1} ~= {1/(VOCAB-1):.4f}')

## 5 · Speed Benchmark (run before the full sweep)

In [ ]:
print('Benchmarking 10 steps per model at n_pairs=32...')
print('='*60)
for mname, (factory, d_model, _, _) in MODEL_REGISTRY.items():
    torch.manual_seed(0)
    model = TinyLM(factory, d_model, VOCAB).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=LR)
    x, y = make_mqar(BATCH, 32, VOCAB)
    if DEVICE=='cuda': torch.cuda.synchronize()
    t0 = time.time()
    for _ in range(10):
        loss = F.cross_entropy(model(x).view(-1,VOCAB), y.view(-1), ignore_index=-100)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    if DEVICE=='cuda': torch.cuda.synchronize()
    dt = (time.time()-t0)/10
    print(f'  {mname:14s}: {dt*1000:6.0f} ms/step  '
          f'-> ~{dt*2800/60:.1f} min for 2800 steps')
    del model, opt; gc.collect()
    if DEVICE=='cuda': torch.cuda.empty_cache()
print()
print('If any model shows multi-second/step, stop and check before running the sweep.')

## 6 · MQAR Capacity Sweep (7 checkpointed cells, n_pairs up to 256)

One cell per n_pairs value. Each saves its checkpoint immediately --
a kernel restart costs at most one cell's work, and `cap_rows` recovers
automatically from disk (see setup cell).

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# n_pairs = 32
# ══════════════════════════════════════════════════════════════════════
n = 32
st = steps_for_n(n)
print(f'{"="*65}')
print(f' n_pairs={n}   steps={st:,}')
print(f'{"="*65}')

t_cell = time.time()
for mname, (factory, d_model, _, _) in MODEL_REGISTRY.items():
    accs = []
    print(f'\n  [{mname}]  d_model={d_model}')
    for seed in SEEDS:
        t0  = time.time()
        acc = run_mqar(factory, d_model, n, steps=st, batch=BATCH, lr=LR, seed=seed)
        dt  = time.time() - t0
        accs.append(round(acc, 4))
        print(f'    seed={seed:4d}: acc={acc:.4f}   ({dt:.0f}s)')
    mean, std = float(np.mean(accs)), float(np.std(accs))
    print(f'    {"-"*40}')
    print(f'    MEAN={mean:.4f}  STD={std:.4f}  seeds={accs}')
    cap_rows.append({'n_pairs': n, 'model': mname,
                     'mean': round(mean,4), 'std': round(std,4),
                     'seeds': str(accs)})

elapsed = (time.time() - t_cell) / 60
print(f'\n  n_pairs={n} done in {elapsed:.1f} min')

pd.DataFrame([r for r in cap_rows if r['n_pairs']==n]).to_csv(
    OUT/'logs'/f'exp1_n{n}.csv', index=False)
print(f'  Checkpoint saved: exp1_n{n}.csv')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# n_pairs = 64
# ══════════════════════════════════════════════════════════════════════
n = 64
st = steps_for_n(n)
print(f'{"="*65}')
print(f' n_pairs={n}   steps={st:,}')
print(f'{"="*65}')

t_cell = time.time()
for mname, (factory, d_model, _, _) in MODEL_REGISTRY.items():
    accs = []
    print(f'\n  [{mname}]  d_model={d_model}')
    for seed in SEEDS:
        t0  = time.time()
        acc = run_mqar(factory, d_model, n, steps=st, batch=BATCH, lr=LR, seed=seed)
        dt  = time.time() - t0
        accs.append(round(acc, 4))
        print(f'    seed={seed:4d}: acc={acc:.4f}   ({dt:.0f}s)')
    mean, std = float(np.mean(accs)), float(np.std(accs))
    print(f'    {"-"*40}')
    print(f'    MEAN={mean:.4f}  STD={std:.4f}  seeds={accs}')
    cap_rows.append({'n_pairs': n, 'model': mname,
                     'mean': round(mean,4), 'std': round(std,4),
                     'seeds': str(accs)})

elapsed = (time.time() - t_cell) / 60
print(f'\n  n_pairs={n} done in {elapsed:.1f} min')

pd.DataFrame([r for r in cap_rows if r['n_pairs']==n]).to_csv(
    OUT/'logs'/f'exp1_n{n}.csv', index=False)
print(f'  Checkpoint saved: exp1_n{n}.csv')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# n_pairs = 96
# ══════════════════════════════════════════════════════════════════════
n = 96
st = steps_for_n(n)
print(f'{"="*65}')
print(f' n_pairs={n}   steps={st:,}')
print(f'{"="*65}')

t_cell = time.time()
for mname, (factory, d_model, _, _) in MODEL_REGISTRY.items():
    accs = []
    print(f'\n  [{mname}]  d_model={d_model}')
    for seed in SEEDS:
        t0  = time.time()
        acc = run_mqar(factory, d_model, n, steps=st, batch=BATCH, lr=LR, seed=seed)
        dt  = time.time() - t0
        accs.append(round(acc, 4))
        print(f'    seed={seed:4d}: acc={acc:.4f}   ({dt:.0f}s)')
    mean, std = float(np.mean(accs)), float(np.std(accs))
    print(f'    {"-"*40}')
    print(f'    MEAN={mean:.4f}  STD={std:.4f}  seeds={accs}')
    cap_rows.append({'n_pairs': n, 'model': mname,
                     'mean': round(mean,4), 'std': round(std,4),
                     'seeds': str(accs)})

elapsed = (time.time() - t_cell) / 60
print(f'\n  n_pairs={n} done in {elapsed:.1f} min')

pd.DataFrame([r for r in cap_rows if r['n_pairs']==n]).to_csv(
    OUT/'logs'/f'exp1_n{n}.csv', index=False)
print(f'  Checkpoint saved: exp1_n{n}.csv')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# n_pairs = 128   <- Uniform-VLA d_h boundary
# ══════════════════════════════════════════════════════════════════════
n = 128
st = steps_for_n(n)
print(f'{"="*65}')
print(f' n_pairs={n}   steps={st:,}   <- Uniform-VLA d_h boundary')
print(f'{"="*65}')

t_cell = time.time()
for mname, (factory, d_model, _, _) in MODEL_REGISTRY.items():
    accs = []
    print(f'\n  [{mname}]  d_model={d_model}')
    for seed in SEEDS:
        t0  = time.time()
        acc = run_mqar(factory, d_model, n, steps=st, batch=BATCH, lr=LR, seed=seed)
        dt  = time.time() - t0
        accs.append(round(acc, 4))
        print(f'    seed={seed:4d}: acc={acc:.4f}   ({dt:.0f}s)')
    mean, std = float(np.mean(accs)), float(np.std(accs))
    print(f'    {"-"*40}')
    print(f'    MEAN={mean:.4f}  STD={std:.4f}  seeds={accs}')
    cap_rows.append({'n_pairs': n, 'model': mname,
                     'mean': round(mean,4), 'std': round(std,4),
                     'seeds': str(accs)})

elapsed = (time.time() - t_cell) / 60
print(f'\n  n_pairs={n} done in {elapsed:.1f} min')

pd.DataFrame([r for r in cap_rows if r['n_pairs']==n]).to_csv(
    OUT/'logs'/f'exp1_n{n}.csv', index=False)
print(f'  Checkpoint saved: exp1_n{n}.csv')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# n_pairs = 160   <- past Uniform-VLA boundary
# ══════════════════════════════════════════════════════════════════════
n = 160
st = steps_for_n(n)
print(f'{"="*65}')
print(f' n_pairs={n}   steps={st:,}   <- past Uniform-VLA boundary')
print(f'{"="*65}')

t_cell = time.time()
for mname, (factory, d_model, _, _) in MODEL_REGISTRY.items():
    accs = []
    print(f'\n  [{mname}]  d_model={d_model}')
    for seed in SEEDS:
        t0  = time.time()
        acc = run_mqar(factory, d_model, n, steps=st, batch=BATCH, lr=LR, seed=seed)
        dt  = time.time() - t0
        accs.append(round(acc, 4))
        print(f'    seed={seed:4d}: acc={acc:.4f}   ({dt:.0f}s)')
    mean, std = float(np.mean(accs)), float(np.std(accs))
    print(f'    {"-"*40}')
    print(f'    MEAN={mean:.4f}  STD={std:.4f}  seeds={accs}')
    cap_rows.append({'n_pairs': n, 'model': mname,
                     'mean': round(mean,4), 'std': round(std,4),
                     'seeds': str(accs)})

elapsed = (time.time() - t_cell) / 60
print(f'\n  n_pairs={n} done in {elapsed:.1f} min')

pd.DataFrame([r for r in cap_rows if r['n_pairs']==n]).to_csv(
    OUT/'logs'/f'exp1_n{n}.csv', index=False)
print(f'  Checkpoint saved: exp1_n{n}.csv')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# n_pairs = 200   <- KEY RESULT: roadmap pass/fail point
# ══════════════════════════════════════════════════════════════════════
n = 200
st = steps_for_n(n)
print(f'{"="*65}')
print(f' n_pairs={n}   steps={st:,}   <- KEY RESULT: roadmap pass/fail point')
print(f'{"="*65}')

t_cell = time.time()
for mname, (factory, d_model, _, _) in MODEL_REGISTRY.items():
    accs = []
    print(f'\n  [{mname}]  d_model={d_model}')
    for seed in SEEDS:
        t0  = time.time()
        acc = run_mqar(factory, d_model, n, steps=st, batch=BATCH, lr=LR, seed=seed)
        dt  = time.time() - t0
        accs.append(round(acc, 4))
        print(f'    seed={seed:4d}: acc={acc:.4f}   ({dt:.0f}s)')
    mean, std = float(np.mean(accs)), float(np.std(accs))
    print(f'    {"-"*40}')
    print(f'    MEAN={mean:.4f}  STD={std:.4f}  seeds={accs}')
    cap_rows.append({'n_pairs': n, 'model': mname,
                     'mean': round(mean,4), 'std': round(std,4),
                     'seeds': str(accs)})

elapsed = (time.time() - t_cell) / 60
print(f'\n  n_pairs={n} done in {elapsed:.1f} min')

pd.DataFrame([r for r in cap_rows if r['n_pairs']==n]).to_csv(
    OUT/'logs'/f'exp1_n{n}.csv', index=False)
print(f'  Checkpoint saved: exp1_n{n}.csv')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# n_pairs = 256   <- Hetero-VLA slow-head capacity boundary   +   AGGREGATION
# ══════════════════════════════════════════════════════════════════════
n = 256
st = steps_for_n(n)
print(f'{"="*65}')
print(f' n_pairs={n}   steps={st:,}')
print(f'{"="*65}')

t_cell = time.time()
for mname, (factory, d_model, _, _) in MODEL_REGISTRY.items():
    accs = []
    print(f'\n  [{mname}]  d_model={d_model}')
    for seed in SEEDS:
        t0  = time.time()
        acc = run_mqar(factory, d_model, n, steps=st, batch=BATCH, lr=LR, seed=seed)
        dt  = time.time() - t0
        accs.append(round(acc, 4))
        print(f'    seed={seed:4d}: acc={acc:.4f}   ({dt:.0f}s)')
    mean, std = float(np.mean(accs)), float(np.std(accs))
    print(f'    {"-"*40}')
    print(f'    MEAN={mean:.4f}  STD={std:.4f}  seeds={accs}')
    cap_rows.append({'n_pairs': n, 'model': mname,
                     'mean': round(mean,4), 'std': round(std,4),
                     'seeds': str(accs)})

elapsed = (time.time() - t_cell) / 60
print(f'\n  n_pairs={n} done in {elapsed:.1f} min')

pd.DataFrame([r for r in cap_rows if r['n_pairs']==n]).to_csv(
    OUT/'logs'/f'exp1_n{n}.csv', index=False)
print(f'  Checkpoint saved: exp1_n{n}.csv')

# ── FINAL AGGREGATION ──────────────────────────────────────────────────
print(f'\n{"="*65}')
print(' ALL 7 VALUES OF n_pairs COMPLETE -- building master CSV')
print(f'{"="*65}')

df_cap = pd.DataFrame(cap_rows)
df_cap.to_csv(OUT/'logs'/'exp1_capacity.csv', index=False)

print('\nFull capacity table:')
print(f'  {"n_pairs":>8}  {"Uniform-VLA":>13}  {"Hetero-VLA":>12}  {"DeltaNet":>10}')
print(f'  {"-"*52}')
for nv in N_PAIRS:
    row = {r['model']: r for _, r in df_cap[df_cap.n_pairs==nv].iterrows()}
    u = row.get('Uniform-VLA', {}).get('mean', float('nan'))
    h = row.get('Hetero-VLA',  {}).get('mean', float('nan'))
    d = row.get('DeltaNet',    {}).get('mean', float('nan'))
    print(f'  {nv:>8}:  {u:>13.4f}  {h:>12.4f}  {d:>10.4f}')

print(f'\nSaved: exp1_capacity.csv')

## 7 · Publication-Ready Figure

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))

for mname, (factory, d_model, col, mk) in MODEL_REGISTRY.items():
    g = df_cap[df_cap.model == mname].sort_values('n_pairs')
    cap = CAP_HETERO if mname=='Hetero-VLA' else CAP_UNIF
    ax.errorbar(g.n_pairs, g['mean'], yerr=g['std'],
                color=col, marker=mk, lw=2.5, ms=9, capsize=5, capthick=1.5,
                label=f'{mname} (capacity={cap})')

ax.axvline(UNIF_DH, color=C['uniform_vla'], ls='--', lw=1.8, alpha=0.8,
           label=f'Uniform d_h={UNIF_DH} boundary')
ax.axvline(SLOW_DH, color=C['hetero_vla'], ls='--', lw=1.8, alpha=0.8,
           label=f'Hetero slow d_h={SLOW_DH} boundary')
ax.axhline(0.80, color='#636e72', ls=':', lw=1.5)
ax.text(N_PAIRS[-1]*0.98, 0.82, 'pass (0.80)', ha='right', fontsize=9, color='#636e72')
ax.axhline(1/(VOCAB-1), color='lightgray', ls=':', lw=1.2)

ax.set(title='MQAR Capacity: Real Roadmap Config\n'
             f'Hetero-VLA (cap={CAP_HETERO}) vs Uniform-VLA (cap={CAP_UNIF})',
       xlabel='n_pairs stored', ylabel='Eval accuracy (mean +- std, 3 seeds)')
ax.set_ylim(0, 1.05); ax.legend(fontsize=9, loc='lower left')

g200_u = df_cap[(df_cap.model=='Uniform-VLA') & (df_cap.n_pairs==200)]
g200_h = df_cap[(df_cap.model=='Hetero-VLA')  & (df_cap.n_pairs==200)]
if not g200_u.empty and not g200_h.empty:
    acc_u, acc_h = g200_u['mean'].values[0], g200_h['mean'].values[0]
    ax.annotate(f'n=200: Hetero={acc_h:.2f}  Uniform={acc_u:.2f}',
                xy=(200, max(acc_h, acc_u)), xytext=(-100, 30),
                textcoords='offset points', fontsize=10,
                arrowprops=dict(arrowstyle='->', color='black', lw=1.2),
                bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                          edgecolor='gray', alpha=0.9))

plt.tight_layout()
plt.savefig(OUT/'plots'/'paper_figure_nb12c.pdf', bbox_inches='tight')
plt.savefig(OUT/'plots'/'paper_figure_nb12c.png', dpi=200, bbox_inches='tight')
plt.show()
print('Saved: paper_figure_nb12c.{pdf,png}')

## 8 · Roadmap Claim: Pass/Fail

In [ ]:
print('='*70)
print('NOTEBOOK 12c RESULTS  --  REAL ROADMAP CONFIG')
print('='*70)
print(f'  Hetero-VLA capacity = {CAP_HETERO}  ({FAST_H}x{FAST_DH} fast + {SLOW_H}x{SLOW_DH} slow)')
print(f'  Uniform-VLA capacity = {CAP_UNIF}  ({UNIF_H}x{UNIF_DH})')
print()

for n in N_PAIRS:
    row = {r['model']: r for _, r in df_cap[df_cap.n_pairs==n].iterrows()}
    u = row.get('Uniform-VLA',{}).get('mean', float('nan'))
    h = row.get('Hetero-VLA', {}).get('mean', float('nan'))
    d = row.get('DeltaNet',   {}).get('mean', float('nan'))
    marker = ' <- KEY' if n in [128, 200] else ''
    print(f'  n={n:4d}:  Uniform={u:.4f}  Hetero={h:.4f}  DeltaNet={d:.4f}{marker}')

g200_u = df_cap[(df_cap.model=='Uniform-VLA') & (df_cap.n_pairs==200)]
g200_h = df_cap[(df_cap.model=='Hetero-VLA')  & (df_cap.n_pairs==200)]
print()
print('ROADMAP CLAIM CHECK @ n_pairs=200:')
if not g200_u.empty and not g200_h.empty:
    acc_u, acc_h = g200_u['mean'].values[0], g200_h['mean'].values[0]
    pass_h = acc_h > 0.80
    print(f'  Hetero-VLA : {acc_h:.4f}   target > 0.80   {"PASS" if pass_h else "FAIL"}')
    print(f'  Uniform-VLA: {acc_u:.4f}   (expected to be well below Hetero by n=200)')
    if pass_h and acc_h > acc_u + 0.3:
        print()
        print('  ROADMAP CENTRAL CLAIM CONFIRMED:')
        print('  Heterogeneous capacity scaling (1152 vs 1024) closes the')
        print('  associative-recall gap that uniform-head VLA cannot close.')
        print('  This is now safe to use as the basis for: forget-gate ablation,')
        print('  BPC re-run at this same config, third-party benchmarks, paper draft.')
    else:
        print()
        print('  Claim NOT yet confirmed at this n_pairs. Do not proceed to')
        print('  forget-gate ablation or external claims until this resolves --')
        print('  check the per-step progress logs above for a stuck-at-random pattern.')
else:
    print('  n=200 data missing -- re-run the n=200 sweep cell.')

manifest = {
    'hetero_design': {'fast': f'{FAST_H}x{FAST_DH}', 'slow': f'{SLOW_H}x{SLOW_DH}',
                       'cap': CAP_HETERO, 'd_model': D_HETERO},
    'uniform_design': {'heads': f'{UNIF_H}x{UNIF_DH}', 'cap': CAP_UNIF, 'd_model': D_UNIF},
    'capacity_results': df_cap.to_dict('records'),
}
with open(OUT/'manifest.json','w') as f:
    json.dump(manifest, f, indent=2)
print(f'\nAll files: {OUT}')